# Fluxo de Dados — carteira_auto v0.2.1

Este notebook documenta o fluxo de dados do sistema:
**Fontes externas → Fetchers → DataLake (SQLite) → Consulta**

Cobre:
1. Configuracao e Settings
2. Os 7 Fetchers (BCB, Yahoo, IBGE, FRED, CVM, Tesouro, DDM)
3. DataLake (PriceLake, MacroLake, FundamentalsLake, NewsLake)
4. IngestNodes (ingestao automatizada via pipeline)

## 1. Configuracao e Settings

Toda configuracao do sistema fica centralizada em `config/settings.py`.
Cada fetcher tem seu dataclass de config com timeout, rate limit e cache TTL.

In [ ]:
from carteira_auto.config import settings

# Paths do sistema
print(f"Raiz do projeto: {settings.paths.ROOT_DIR}")
print(f"DataLake: {settings.paths.LAKE_DIR}")
print(f"Planilha: {settings.paths.PORTFOLIO_FILE}")

# Config de fetchers
print(f"\nYahoo timeout: {settings.yahoo.TIMEOUT}s")
print(f"BCB rate limit: {settings.bcb.RATE_LIMIT_CALLS}/min")
print(f"Portfolio risk-free daily: {settings.portfolio.RISK_FREE_DAILY}")

In [ ]:
from carteira_auto.config.constants import constants

# Series BCB disponiveis
print("Series BCB registradas:")
for nome, codigo in constants.BCB_SERIES_CODES.items():
    print(f"  {nome}: SGS {codigo}")

## 2. Fetchers — Coleta de Dados Externos

Cada fetcher encapsula uma fonte de dados com retry, rate limit e cache.
Todos retornam `pd.DataFrame` com colunas padronizadas.

| Fetcher | Fonte | Autenticacao |
|---------|-------|--------------|
| BCBFetcher | Banco Central (SGS) | Livre |
| IBGEFetcher | IBGE (SIDRA) | Livre |
| YahooFinanceFetcher | Yahoo Finance | Livre |
| FREDFetcher | Federal Reserve | API key (gratuita) |
| CVMFetcher | CVM Dados Abertos | Livre |
| TesouroDiretoFetcher | Tesouro Nacional | Livre |
| DDMFetcher | Dados de Mercado | API key |

### 2.1 BCBFetcher — Banco Central do Brasil (SGS API)

Busca indicadores macroeconomicos via API SGS do BCB.
Sem autenticacao. Rate limit: 30 req/min. Cache: 1h.

In [ ]:
from carteira_auto.data.fetchers import BCBFetcher

bcb = BCBFetcher()

# Metodos especificos (retornam DataFrame com ['data', 'valor'])
selic = bcb.get_selic(period_days=30)
print(f"Selic (ultimos 30 dias): {len(selic)} registros")
print(selic.tail(3))

In [ ]:
# Outros indicadores disponiveis
cdi = bcb.get_cdi(period_days=30)
ipca = bcb.get_ipca(period_days=365)
igpm = bcb.get_igpm(period_days=365)
ptax = bcb.get_ptax(period_days=30)  # PTAX compra USD/BRL

print(f"CDI: {len(cdi)} registros")
print(f"IPCA: {len(ipca)} registros")
print(f"IGP-M: {len(igpm)} registros")
print(f"PTAX: {len(ptax)} registros")

In [ ]:
# Metodo generico — busca qualquer serie por codigo SGS
from datetime import date, timedelta

start = date.today() - timedelta(days=365)
divida_pib = bcb.get_indicator(13762, start_date=start)  # Divida bruta/PIB %
print(f"Divida bruta/PIB: {divida_pib['valor'].iloc[-1]:.1f}%")

### 2.2 IBGEFetcher — IBGE (SIDRA API)

Busca indicadores estruturais do IBGE via API SIDRA.
Sem autenticacao. Rate limit: 30 req/min. Cache: 2h.

In [ ]:
from carteira_auto.data.fetchers import IBGEFetcher

ibge = IBGEFetcher()

# PIB trimestral
pib = ibge.get_pib(quarters=8)
print(f"PIB: {len(pib)} trimestres")
print(pib.tail(3))

In [ ]:
# Desemprego e IPCA detalhado
desemprego = ibge.get_unemployment(quarters=8)
ipca_detalhado = ibge.get_ipca(months=12)

print(f"Desemprego: {len(desemprego)} registros")
print(f"IPCA detalhado: {len(ipca_detalhado)} registros")

### 2.3 YahooFinanceFetcher — Yahoo Finance

Busca precos, fundamentos e dividendos via yfinance.
Sem autenticacao. Rate limit: 30 req/min. Cache: 5min (precos) / 24h (historico).

In [ ]:
from carteira_auto.data.fetchers import YahooFinanceFetcher

yahoo = YahooFinanceFetcher()

# Precos atuais (batch)
precos = yahoo.get_multiple_prices(["PETR4.SA", "VALE3.SA", "ITUB4.SA"])
print("Precos atuais:")
for ticker, preco in precos.items():
    print(f"  {ticker}: R${preco:.2f}" if preco else f"  {ticker}: indisponivel")

In [ ]:
# Historico OHLCV
historico = yahoo.get_historical_price_data(
    ["PETR4.SA", "VALE3.SA"], period="3mo", interval="1d"
)
print(f"Historico: {historico.shape}")
print(historico.tail(3))

In [ ]:
# Dados fundamentalistas (batch)
info = yahoo.get_batch_info(
    ["PETR4.SA", "VALE3.SA"],
    fields=["trailingPE", "priceToBook", "dividendYield", "returnOnEquity"]
)
print("Fundamentos:")
print(info)

### 2.4 FREDFetcher — Federal Reserve Economic Data

Busca indicadores economicos dos EUA via API FRED.
Requer `FRED_API_KEY` no `.env`. Rate limit: 120 req/min. Cache: 24h.

In [ ]:
# Requer FRED_API_KEY configurada no .env
import os
FRED_OK = bool(os.getenv("FRED_API_KEY"))
print(f"FRED API key configurada: {FRED_OK}")

if FRED_OK:
    from carteira_auto.data.fetchers import FREDFetcher
    fred = FREDFetcher()

    # Metodos especificos
    fed_funds = fred.get_fed_funds_rate()
    treasury_10y = fred.get_treasury_10y()
    vix = fred.get_vix()

    print(f"Fed Funds: {fed_funds['value'].iloc[-1]:.2f}%")
    print(f"Treasury 10y: {treasury_10y['value'].iloc[-1]:.2f}%")
    print(f"VIX: {vix['value'].iloc[-1]:.1f}")

    # Metodo generico
    cpi = fred.get_series("CPIAUCSL")  # CPI All Urban Consumers
    print(f"CPI US: {len(cpi)} registros")

### 2.5 CVMFetcher — Comissao de Valores Mobiliarios

Busca demonstracoes financeiras e cadastros via dados abertos da CVM.
Sem autenticacao. Rate limit: 30 req/min. Cache: 24h.

In [ ]:
from carteira_auto.data.fetchers import CVMFetcher

cvm = CVMFetcher()

# Cadastro de empresas
cadastro = cvm.get_company_registry()
print(f"Empresas registradas na CVM: {len(cadastro)}")
print(cadastro[["DENOM_SOCIAL", "CD_CVM"]].head(5))

In [ ]:
# Demonstracoes financeiras por ticker
# Nota: depende do endpoint CVM estar disponivel
try:
    dre = cvm.get_dfp_by_ticker("PETR4", year=2023, statement="DRE")
    print(f"DRE Petrobras 2023: {len(dre)} linhas")
except Exception as e:
    print(f"CVM indisponivel: {e}")

### 2.6 TesouroDiretoFetcher — Tesouro Nacional

Busca taxas e titulos do Tesouro Direto.
Sem autenticacao. Rate limit: 30 req/min. Cache: 1h.

In [ ]:
from carteira_auto.data.fetchers import TesouroDiretoFetcher

tesouro = TesouroDiretoFetcher()

# Taxas atuais
taxas = tesouro.get_current_rates()
print(f"Titulos disponiveis: {len(taxas)}")
print(taxas[["tipo_titulo", "taxa_compra_manha"]].head(5))

In [ ]:
# Curva NTN-B (juros reais)
curva = tesouro.get_ntnb_curve()
print("Curva NTN-B (juros reais):")
print(curva)

### 2.7 DDMFetcher — Dados de Mercado

Busca cotacoes, fundamentos e indicadores via API Dados de Mercado.
Requer `DADOS_MERCADO_API_KEY` no `.env`. Cache: 24h.

In [ ]:
DDM_OK = bool(os.getenv("DADOS_MERCADO_API_KEY"))
print(f"DDM API key configurada: {DDM_OK}")

if DDM_OK:
    from carteira_auto.data.fetchers import DDMFetcher
    ddm = DDMFetcher()

    # Indices de mercado
    indices = ddm.get_market_indices()
    print(f"Indices: {len(indices)} registros")

    # Cotacao individual
    cotacao = ddm.get_quotations("PETR4", period_init="2024-01-01")
    print(f"PETR4 cotacoes: {len(cotacao)} registros")

## 3. DataLake — Persistencia SQLite

O DataLake persiste dados historicos em 4 bancos SQLite independentes.
E o single source of truth para dados historicos do sistema.

| Sub-lake | Arquivo | Conteudo |
|----------|---------|----------|
| PriceLake | prices.db | Series OHLCV por ticker |
| MacroLake | macro.db | Indicadores macroeconomicos |
| FundamentalsLake | fundamentals.db | Dados fundamentalistas |
| NewsLake | news.db | Artigos e sentimento |

In [ ]:
from pathlib import Path
import tempfile
from carteira_auto.data.lake import DataLake

# Usar diretorio temporario para demonstracao
demo_dir = Path(tempfile.mkdtemp()) / "demo_lake"
lake = DataLake(demo_dir)

print(f"DataLake em: {demo_dir}")
print(f"Sub-lakes: prices, macro, fundamentals, news")

In [ ]:
import pandas as pd

# Armazenar precos no PriceLake
precos_df = pd.DataFrame({
    "Open": [35.0, 35.5],
    "High": [36.0, 36.2],
    "Low": [34.5, 35.0],
    "Close": [35.5, 36.0],
    "Volume": [1000000, 1200000],
}, index=pd.date_range("2024-01-01", periods=2, freq="B"))

lake.prices.store_prices("PETR4.SA", precos_df)
print("Precos armazenados no PriceLake")

# Consultar precos
resultado = lake.prices.get_prices("PETR4.SA")
print(f"Registros recuperados: {len(resultado)}")
print(resultado)

In [ ]:
# Armazenar indicador macro
selic_df = pd.DataFrame({
    "data": pd.date_range("2024-01-01", periods=5, freq="B"),
    "valor": [13.75, 13.75, 13.75, 13.75, 13.75],
})

lake.macro.store_indicator("selic", selic_df)
print("Selic armazenada no MacroLake")

# Consultar indicador
selic_lake = lake.macro.get_indicator("selic")
print(f"Selic do lake: {len(selic_lake)} registros")

In [ ]:
# Limpeza
import shutil
shutil.rmtree(demo_dir, ignore_errors=True)
print("Demo lake limpo")

## 4. IngestNodes — Ingestao Automatizada

IngestNodes sao Nodes do DAG que orquestram a ingestao de dados
dos fetchers para o DataLake. Sao executados via CLI ou pipeline.

```bash
# Via CLI
python -m carteira_auto ingest --mode daily
python -m carteira_auto ingest --mode full

# Via pipelines individuais
python -m carteira_auto run ingest-prices
python -m carteira_auto run ingest-macro
python -m carteira_auto run ingest-fundamentals
```

In [ ]:
from carteira_auto.core.nodes.ingest_nodes import (
    IngestPricesNode,
    IngestMacroNode,
    IngestFundamentalsNode,
    IngestNewsNode,
)

# Cada IngestNode e um Node do DAG sem dependencias
for node_cls in [IngestPricesNode, IngestMacroNode, IngestFundamentalsNode, IngestNewsNode]:
    node = node_cls()
    print(f"{node.name}: deps={node.dependencies}")

In [ ]:
# Para executar via DAGEngine (sem CLI):
from carteira_auto.core.registry import create_engine, get_terminal_node

engine = create_engine()

# Dry-run para ver o plano de execucao
plan = engine.dry_run(get_terminal_node("ingest-macro"))
print("Plano de ingestao macro:")
for step in plan:
    print(f"  -> {step}")

## Resumo

O fluxo de dados do carteira_auto segue o padrao:

```
Fontes externas (APIs)
       |
       v
Fetchers (7 implementados) — @retry, @rate_limit, @cache
       |
       v
DataLake (4 sub-lakes SQLite) — persistencia historica
       |
       v
PipelineContext — contexto compartilhado entre nodes
       |
       v
Analyzers / Exporters / Alerts
```

Proximos notebooks:
- `02_orchestration.ipynb` — DAGEngine, Node, PipelineContext, Registry
- `03_processing.ipynb` — Models, Analyzers, Error Handling, Alerts